# 09 — Ablation Studies

This notebook runs the mandatory **ablation studies**: controlled experiments in which one component is removed or changed while everything else is kept fixed, to measure its real contribution. As stressed in the course, ablations are the most direct way to understand *why* a deep model works and to avoid attributing gains to the wrong component.

Experiments are run on the best models from notebooks 04–08 with a common protocol (same split, budget, metrics) and **several random seeds**, so that differences can be distinguished from training noise.

> **Status: skeleton.** Only the introduction, objectives and work preview are written. Code and results will be added when the phase is developed.

> **Estado de ejecución verificado · 20/11/2026.** Plan de trabajo: todavía sin implementación ni resultados de entrenamiento. Las ablaciones de redes pendientes no se presentan como realizadas.

## Ablaciones para aprender, no para perseguir el mejor resultado

La interpretación transversal de «don't look for performance» es que debemos poder explicar los experimentos. No significa dejar de medir ni relajar la evaluación. Para cada ablación: hipótesis, configuración de referencia, único cambio, presupuesto, semillas y resultado con variabilidad. Priorizar pocas comparaciones interpretables.

Ejemplos por vía: CNN con/sin aumento; LSTM con último estado frente a pooling con atención; encoder preentrenado congelado frente a fine-tuning; arquitectura idéntica con pesos aleatorios frente a preentrenados. Si se compara origen supervisado y autosupervisado con datos o presupuestos distintos, no atribuir toda la diferencia únicamente al objetivo de preentrenamiento. No es necesario ejecutar toda esta lista; se escogerá un subconjunto justificado. No usar el test para escoger la variante.


## Protocolo revisado antes del entrenamiento · 20/11/2026

La referencia común es `config/experimental_protocol_v1.json`. Es un diseño fijado antes de entrenar las redes, no una lista de resultados obtenidos. La versión equilibrada de 18.000 fotos se conserva como experimento exploratorio. Sus métricas describen las doce clases seleccionadas con ese muestreo; no estiman automáticamente el comportamiento sobre el catálogo original.

**Por defecto:** entropía cruzada sin pesos, sin aumentos y con normalización de train a la resolución de entrada. Pesos de clase y aumentos se activarán como ablaciones explícitas. Los pesos guardados en 02 son una opción calculada, no una instrucción de aplicarlos siempre. Cada experimento registrará resolución, preprocesamiento, IDs de productos, configuración, semilla y presupuesto.

**Métricas fijadas:** macro-F1 primaria; accuracy secundaria; precisión, recall, F1 y soporte por clase; matriz de confusión en recuentos y normalizada por filas. Comparar parámetros entrenables, tiempo de entrenamiento, latencia por imagen, memoria pico y hardware. Selección por macro-F1 de validation; early stopping por pérdida de validation; tres semillas (42, 123 y 2026). El test permanece cerrado hasta congelar decisiones; intervalos finales mediante remuestreo de grupos de producto, no imágenes independientes. Los baselines deterministas no necesitan simular tres entrenamientos diferentes.

### Matriz de hipótesis y experimentos prioritarios

| ID | Contraste fijado | Qué permanece fijo | Evidencia principal |
|---|---|---|---|
| H1 | CNN frente a Dense | RGB64, mismos IDs, parámetros ±10 %, pérdida sin pesos | Diferencia de macro-F1 y F1 por clase; coste |
| H2a | Normalización sí/no | Una CNN y resolución, sin aumentos | Macro-F1 y curvas |
| H2b | Aumentos sí/no | La misma CNN normalizada | Macro-F1 y brecha train/validation |
| H3 | Pesos aleatorios/preentrenados | Backbone, entrada y particiones | Macro-F1, convergencia y coste |
| H4 | Pérdida sin/con pesos | Una CNN y todo su protocolo | Macro-F1 y recall por clase |
| H5 | Train equilibrado/proporcional | Presupuesto de imágenes comparable y misma evaluación | Recall minoritario y macro-F1 |
| H6 | Columnas/filas; parches como variante | RGB64, IDs, tamaño oculto y capas | Macro-F1, forma de entrada y coste |

Estos experimentos están **diseñados, no ejecutados**. No ampliar la búsqueda tras ver el test. Diferencias pequeñas o negativas se informarán con su variación entre semillas; no se interpretarán como mejoras solo por el signo de una ejecución.

**H5, equilibrio de entrenamiento:** formar dos subconjuntos exclusivamente de train, con grupos completos y presupuesto de unas 3.600 imágenes: uno aproximadamente equilibrado y otro proporcional a las frecuencias del pool elegible antes del cupo. Orden de grupos determinista; documentar el número real alcanzado y cualquier límite de disponibilidad. No mover grupos de validation/test. Mantener resolución, arquitectura, pérdida sin pesos, regla de parada y evaluación fija. Esto aísla sensibilidad al muestreo de train bajo una evaluación equilibrada; no convierte el test en un catálogo natural.

**Sensibilidad de la evaluación:** a partir de las mismas predicciones, informar separadamente métricas equilibradas y métricas ponderadas con prevalencias fijadas a partir del pool elegible anterior al cupo. Es un análisis bajo supuesto de cambio de prevalencia y estabilidad dentro de cada clase; no demuestra rendimiento real del catálogo. El pool conserva solo las clases elegidas y las reglas de calidad, no todo H&M. Las prevalencias se fijan por metadatos antes de analizar errores, nunca para favorecer al modelo.

**Réplica con prevalencia natural, posterior y separada:** crear una nueva versión dividiendo globalmente todos los grupos elegibles antes de limitar entrenamiento, manteniendo validation/test sin equilibrarlos. No añadir ahora fotos excluidas al train de v1: podrían estar conectadas con sus grupos de evaluación. Una nueva réplica no sobrescribe v1 y requiere regenerar y verificar sus manifiestos. La combinación de reponderación y H5 es el primer paso viable; la réplica completa queda pendiente, sin afirmar que ya se haya realizado.


## Objectives

- Define a fixed experimental protocol: base configuration, training budget, seeds and metrics.
- Ablate regularisation: dropout on/off and rate, weight decay, batch normalisation on/off.
- Ablate data: augmentation on/off, input resolution, grayscale vs. RGB, class weights on/off.
- Ablate architecture: depth, width, skip connections, pooling type.
- Ablate optimisation: SGD vs. momentum vs. Nesterov vs. Adam, learning-rate schedule, batch size.
- Ablate initialisation: pretrained vs. from scratch, frozen vs. fine-tuned.
- Summarise effects with mean ± standard deviation across seeds and discuss which components matter.

## Preview of the work

1. **Protocol and experiment registry.** configuration dictionaries and a results table in `results/metrics/`.
2. **Regularisation ablations.** dropout, weight decay, BatchNorm.
3. **Data ablations.** augmentation, resolution, colour, class weighting.
4. **Architecture ablations.** depth, width, residual connections.
5. **Optimisation ablations.** optimiser, learning rate, batch size.
6. **Pretraining ablations.** ImageNet weights vs. random initialisation.
7. **Summary table and figures.** effect sizes with uncertainty across seeds; conclusions.

## Inputs, outputs and course references

| | |
|---|---|
| **Inputs** | Best configurations from notebooks 04–08 |
| **Outputs** | Ablation table and figures in `results/metrics/` and `results/figures/` |
| **Course material** | `PRDL_MLLB_FeedForward.pptx` (ablation studies, dropout), `MLLB_TransferLearning_Image_and-Text.pptx` (batch normalisation), `Notebooks_GuideLines_for_DeepLearnig_Report.pptx` |

## Notes and open decisions

- Computational budget on Colab is limited: prioritise a small number of ablations with several seeds over many single-seed runs.